In [2]:
import pandas as pd
import psutil
import time
# This is for ex2.2
def monitor_resources():
    memory_usage = psutil.virtual_memory().percent
    cpu_usage = psutil.cpu_percent(interval=1)
    return f"CPU Usage: {cpu_usage}% | Memory Usage: {memory_usage}%"

def analyze_flight_delays(file_paths):

    start_time = time.time()
    data_frames = [pd.read_csv(file) for file in file_paths]
    end_time = time.time()
    data = pd.concat(data_frames, ignore_index=True)
    
    # print(f"Data loading and concatenation completed in {end_time - start_time:.2f} seconds.")
    # print(f"Total data size: {data.shape[0]} rows")

    data['IsLate'] = data['ArrTime'] > data['CRSArrTime']
    most_commonly_late_carrier = data[data['IsLate']]['UniqueCarrier'].value_counts().idxmax()

    data['WeatherDelayed'] = data['WeatherDelay'] > 0
    most_commonly_late_origins = data[data['WeatherDelayed']]['Origin'].value_counts().nlargest(3).index.tolist()

    data['TotalDelay'] = data['ArrTime'] - data['CRSArrTime']
    longest_delay_per_carrier = data.groupby('UniqueCarrier')['TotalDelay'].max()

    return most_commonly_late_carrier, most_commonly_late_origins, longest_delay_per_carrier

# Specify the paths to data files
file_paths = ['2008.csv']

full_start_time = time.time()

results = analyze_flight_delays(file_paths)

resource_usage = monitor_resources()

full_end_time = time.time()
# print(f"Total analysis completed in {full_end_time - full_start_time:.2f} seconds.")

# Output results and resource usage
# print("Most Commonly Late Carrier:", results[0])
# print("Most Commonly Late Origins Due to Bad Weather:", results[1])
# print("Longest Delay for Each Carrier:\n", results[2])
# print("Resource Usage:", resource_usage)


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
# This is for ex2.3
def analyze_flight_pattern(file_paths):
    start_time = time.time()
    data_frames = [pd.read_csv(file) for file in file_paths]
    end_time = time.time()
    data = pd.concat(data_frames, ignore_index=True)
    data = data.fillna(0)
    
    print(f"Data loading and concatenation completed in {end_time - start_time:.2f} seconds.")
    print(f"Total data size: {data.shape[0]} rows")

    # x is DayOfWeek、DepTime 、CRSDepTime 、ArrTime 、CRSArrTime 、UniqueCarrier.
    X = data[['DayOfWeek', 'DepTime', 'CRSDepTime', 'ArrTime', 'CRSArrTime']]
    y = data['DepDelay']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Mean Squared Error: {mse}")
    
    coefficients = model.coef_

    feature_names = X.columns.tolist()

    for i, coef in enumerate(coefficients):
        print(f"{feature_names[i]}: {coef}")
        
    formula = "DepDelay = " + " + ".join([f"{coef:.2f} * {feature}" for feature, coef in zip(feature_names, coefficients)])
    print(formula)

# Specify the paths to data files
file_paths = ['2008.csv'] 

full_start_time = time.time()

results = analyze_flight_pattern(file_paths)

resource_usage = monitor_resources()

full_end_time = time.time()
print(f"Total analysis completed in {full_end_time - full_start_time:.2f} seconds.")


Data loading and concatenation completed in 48.34 seconds.
Total data size: 7009728 rows
Mean Squared Error: 1141.813381287689
DayOfWeek: 0.2357414260450138
DepTime: 0.04563992750920627
CRSDepTime: -0.034511328753667254
ArrTime: -0.022312792933074546
CRSArrTime: 0.02044026968759575
DepDelay = 0.24 * DayOfWeek + 0.05 * DepTime + -0.03 * CRSDepTime + -0.02 * ArrTime + 0.02 * CRSArrTime
Total analysis completed in 83.87 seconds.
